In [1]:
import os
		
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, BertTokenizer, BertForSequenceClassification
from datasets import load_dataset

d:\miniconda3\envs\PyTorch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
dataset = load_dataset("csv", data_files="./ChnSentiCorp_htl_all.csv", split="train")
dataset = dataset.filter(lambda x: x["review"] is not None)
dataset

Dataset({
    features: ['label', 'review'],
    num_rows: 7765
})

In [9]:
datasets = dataset.train_test_split(test_size=0.1)
datasets

DatasetDict({
    train: Dataset({
        features: ['label', 'review'],
        num_rows: 6988
    })
    test: Dataset({
        features: ['label', 'review'],
        num_rows: 777
    })
})

In [10]:
datasets['train'][:3]

{'label': [1, 1, 1],
 'review': ['8月份在赛格尔住了几天，住过208元的商务间和238元的豪华单人间，感觉如下：优点：1、地理位置很好，走路至解放碑不到5分钟；2、服务较好，换房服务及时，可以延到下午2点退房；3、房间面积很大（在拐角的商务间面积相比要小些）。缺点：1、房间装修一般，木地板，虽然面积不小，但让人感觉装修品位很差；另外电视相比房间面积太小，躺在床上看电视看不清楚；2、总共有33层楼，却只有4部电梯，等电梯时间太长；3、酒店7楼餐厅饭菜种类较少，味道一般，还不提供送餐服务；4、卫生间很小，淋浴的喷头很差，洗澡不舒服；5、房间吧台小吃价格比其他酒店偏贵，洗衣费也偏贵（相比解放碑附近我住过的其他酒店）。总体来说这家酒店一般，不过从房价和地理位置综合考虑，我给它打3.5分。',
  '晚上蚊子很多，本人及住在另一间房的朋友都被蚊子咬醒，在房间内找不到驱蚊香。:(',
  '服务算可以设施老旧空调太吵在济南，也没别的好选择']}

In [18]:
datasets['train']['label'][:3], type(datasets['train']['label'][0])

([1, 1, 1], int)

In [11]:
import torch

tokenizer = BertTokenizer.from_pretrained("hfl/rbt3")

def process_function(examples):
    tokenized_examples = tokenizer(examples["review"], max_length=128, truncation=True)
    tokenized_examples["labels"] = examples["label"]
    return tokenized_examples

tokenized_datasets = datasets.map(process_function, batched=True, remove_columns=datasets["train"].column_names)
tokenized_datasets

Map: 100%|██████████| 777/777 [00:00<00:00, 1606.96 examples/s]


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 6988
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 777
    })
})

In [19]:
tokenized_datasets['train']['labels'][:3]

[1, 1, 1]

In [20]:
model = BertForSequenceClassification.from_pretrained("hfl/rbt3")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at hfl/rbt3 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [21]:
import evaluate

acc_metric = evaluate.load("accuracy")
f1_metirc = evaluate.load("f1")

In [22]:
def eval_metric(eval_predict):
    predictions, labels = eval_predict
    predictions = predictions.argmax(axis=-1)
    acc = acc_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metirc.compute(predictions=predictions, references=labels)
    acc.update(f1)
    return acc

In [23]:
train_args = TrainingArguments(output_dir="./checkpoints",      # 输出文件夹
                               per_device_train_batch_size=64,  # 训练时的batch_size
                               per_device_eval_batch_size=128,  # 验证时的batch_size
                               logging_steps=50,                # log 打印的频率
                               eval_strategy="steps",           # 评估策略
                               save_total_limit=3,              # 最大保存数
                               learning_rate=2e-5,              # 学习率
                               weight_decay=0.01,               # weight_decay
                               metric_for_best_model="f1",      # 设定评估指标
                               load_best_model_at_end=True)     # 训练完成后加载最优模型

In [24]:
hasattr(train_args, "_n_gpu")
train_args.__dict__

{'output_dir': './checkpoints',
 'overwrite_output_dir': False,
 'do_train': False,
 'do_eval': True,
 'do_predict': False,
 'eval_strategy': <IntervalStrategy.STEPS: 'steps'>,
 'prediction_loss_only': False,
 'per_device_train_batch_size': 64,
 'per_device_eval_batch_size': 128,
 'per_gpu_train_batch_size': None,
 'per_gpu_eval_batch_size': None,
 'gradient_accumulation_steps': 1,
 'eval_accumulation_steps': None,
 'eval_delay': 0,
 'torch_empty_cache_steps': None,
 'learning_rate': 2e-05,
 'weight_decay': 0.01,
 'adam_beta1': 0.9,
 'adam_beta2': 0.999,
 'adam_epsilon': 1e-08,
 'max_grad_norm': 1.0,
 'num_train_epochs': 3.0,
 'max_steps': -1,
 'lr_scheduler_type': <SchedulerType.LINEAR: 'linear'>,
 'lr_scheduler_kwargs': {},
 'warmup_ratio': 0.0,
 'warmup_steps': 0,
 'log_level': 'passive',
 'log_level_replica': 'warning',
 'log_on_each_node': True,
 'logging_dir': './checkpoints\\runs\\Dec23_13-45-08_whtcc_yuchangle',
 'logging_strategy': <IntervalStrategy.STEPS: 'steps'>,
 'logging_

In [25]:
from transformers import DataCollatorWithPadding
trainer = Trainer(model=model, 
                  args=train_args, 
                  train_dataset=tokenized_datasets["train"], 
                  eval_dataset=tokenized_datasets["test"], 
                  data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
                  compute_metrics=eval_metric)

In [26]:
trainer.train()

Step,Training Loss,Validation Loss,Accuracy,F1
50,0.525600,0.388237,0.824968,0.873134
100,0.334200,0.333835,0.853282,0.889320
150,0.269000,0.296224,0.881596,0.913534
200,0.276400,0.297947,0.879022,0.913603
250,0.241600,0.295524,0.877735,0.912281
300,0.223600,0.284652,0.884170,0.915730


TrainOutput(global_step=330, training_loss=0.30440277619795364, metrics={'train_runtime': 46.7446, 'train_samples_per_second': 448.48, 'train_steps_per_second': 7.06, 'total_flos': 351909933963264.0, 'train_loss': 0.30440277619795364, 'epoch': 3.0})

In [27]:
trainer.evaluate(tokenized_datasets["test"])

{'eval_loss': 0.28388509154319763,
 'eval_accuracy': 0.8828828828828829,
 'eval_f1': 0.9147141518275539,
 'eval_runtime': 0.75,
 'eval_samples_per_second': 1036.03,
 'eval_steps_per_second': 9.334,
 'epoch': 3.0}

In [28]:
sen = "我觉得这家酒店不错，饭很好吃！"
id2_label = {0: "差评！", 1: "好评！"}
model.eval()
with torch.inference_mode():
    inputs = tokenizer(sen, return_tensors="pt")
    inputs = {k: v.cuda() for k, v in inputs.items()}
    logits = model(**inputs).logits
    pred = torch.argmax(logits, dim=-1)
    print(f"输入：{sen}\n模型预测结果:{id2_label.get(pred.item())}")

输入：我觉得这家酒店不错，饭很好吃！
模型预测结果:好评！


In [29]:
from transformers import pipeline

model.config.id2label = id2_label
pipe = pipeline("text-classification", model=model, tokenizer=tokenizer, device=0)

Device set to use cuda:0


In [30]:
pipe(sen)

[{'label': '好评！', 'score': 0.9951068162918091}]